In [3]:
import os.path as osp
import numpy as np
import cv2
import matplotlib.pyplot as plt
import datetime

MAX_AREA = 2048*2448
#Preprocessing
def preprocessing_image(img):
    #convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.multiply(gray, 1.5)
    
    #blur remove noise
    blured1 = cv2.medianBlur(gray,3)
    blured2 = cv2.medianBlur(gray,47)
    divided = np.ma.divide(blured1, blured2).data
    normed = np.uint8(255*divided/divided.max())
    
    #Threshold image
    th, threshed = cv2.threshold(normed, 0, 255,cv2.THRESH_OTSU+  cv2.THRESH_BINARY)
    
    return threshed

#Blur
def blur_color_img(img, kernel_width=5, kernel_height=5, sigma_x=2, sigma_y=2):
    #increse brightness for foreground
    
    img = np.copy(img) # we don't modify the original image
    img[:,:,0] = cv2.GaussianBlur(img[:,:,0], ksize=(kernel_width, kernel_height), sigmaX=sigma_x, sigmaY=sigma_y)
    img[:,:,1] = cv2.GaussianBlur(img[:,:,1], ksize=(kernel_width, kernel_height), sigmaX=sigma_x, sigmaY=sigma_y)
    img[:,:,2] = cv2.GaussianBlur(img[:,:,2], ksize=(kernel_width, kernel_height), sigmaX=sigma_x, sigmaY=sigma_y)
    # blured1 = cv2.medianBlur(img,3)
    # blured2 = cv2.medianBlur(img,51)
    # img = cv2.GaussianBlur(img, (5, 5), 0)
    return img


#background subtraction
def background_subtraction(fg_img, bg_img, diff_threshold=200):
    fg_img = blur_color_img(fg_img)
    bg_img = blur_color_img(bg_img)
    mask = fg_img - bg_img
    mask = np.abs(mask)
    mask = np.mean(mask, axis=2, keepdims=False)
    mask[mask>diff_threshold] = 255
    mask[mask >= diff_threshold] = 0
    mask = mask.astype(np.uint8)
    mask = cv2.medianBlur(mask, 7)
    return mask

#excute
def main(foreground_img, background_img):
    fg_img = (foreground_img) # [h, w, 3]
    bg_img = (background_img) # [h, w, 3]
    mask = background_subtraction(fg_img, bg_img)
    new_fg = np.zeros([fg_img.shape[0], fg_img.shape[1], 4]) # png image --> has 4-dims instead of 3-dims like color image
    new_fg[:,:,:3] = fg_img
    new_fg[:,:,3] = mask
    # cv2.imwrite('mask.jpg', mask)
    # cv2.imwrite('test1.png', new_fg)
    # display_image(mask)
    return mask







In [13]:
#Post Processing
def boudingBox(fg_cropImageRoi, fgMask):
    fg = fg_cropImageRoi.copy()
    contours, hierarchy = cv2.findContours(image=fgMask, mode=cv2.RETR_EXTERNAL, 
                                       method=cv2.CHAIN_APPROX_NONE)
    # draw contours on the original image
    cv2.drawContours(image=fg, contours=contours, contourIdx=-1, 
                    color=(0, 255, 0), thickness=2, lineType=cv2.LINE_AA)
    
    print(contours)
    cv2.imshow('None approximation', fg)
    cv2.waitKey(0)
    cv2.imwrite('test_contour.jpg', fg)
    cv2.destroyAllWindows()
    # cordinate = 0
    # area_get = 0
    # #You can choose type connectivity with value about from 4 to 8
    # connectivity = 4
    # output = cv2.connectedComponentsWithStats(fgMask, connectivity, cv2.CV_32S)
    # (numLabels, labels, stats, centroids) = output
    # for i in  range(0, numLabels):
    #     x = stats[i, cv2.CC_STAT_LEFT]
    #     y = stats[i, cv2.CC_STAT_TOP]
    #     w = stats[i, cv2.CC_STAT_WIDTH]
    #     h = stats[i, cv2.CC_STAT_HEIGHT]
    #     area = stats[i, cv2.CC_STAT_AREA]

    #     # if (0.3*MAX_AREA < area < MAX_AREA)  and (w < fg.shape[0]) and (h < fg.shape[1]):
    #     # cv2.rectangle(fg, (x, y), (x + w, y + h), (0, 255, 0), 10)
    #     # cordinate = [x, y, w, h]
    #     # text = f'({x}, {y})'
    #     # cv2.putText(fg, text, color = (0, 255, 125), org = (x, y-50), fontFace= 1, fontScale= 3, thickness= 5, lineType= 1)

    #     # text2 = f'({x+w}, {y+h})'
    #     # cv2.putText(fg, text2, color = (0, 255, 125), org = (x+w, y+h+50), fontFace= 1, fontScale= 3, thickness= 5, lineType= 1)
    #     # area_get = area
    
    # # cv2.imshow('Roi Image 2' , fg)
    # # time = datetime.datetime.now()
    # # cv2.imwrite("test_re.png", fg)
    # return cordinate, fg, area_get


In [14]:

start_time = datetime.datetime.now()
# Background
img_bg = cv2.imread('../data\\data_2024_04_20\\9.png', cv2.IMREAD_COLOR)
# Object
obj_path = '../data\\data_2024_04_20\\2_horizontal.png'
img_obj = cv2.imread(obj_path , cv2.IMREAD_COLOR)
# Mask
fmask = main(img_obj, img_bg)
cv2.imwrite(f'../data/mask/{osp.basename(obj_path)}', fmask)
# Return Object
fg = boudingBox(img_obj, fgMask= fmask)
# cv2.imwrite(f'../data/fg/{osp.basename(obj_path)}', fg)
# print(f'The information of object {coordinate}, {area}')
# # Crop Working Space
# x, y, w, h = coordinate
# img_crop = img_obj[y: y+h, x: x+w]
# print(f'Time to process step 1 {datetime.datetime.now() - start_time}')
# # Test save
# cv2.imwrite(f'../data/boudingbox/{osp.basename(obj_path)}', img_crop)


(array([[[0, 0]],

       [[0, 1]],

       [[0, 2]],

       ...,

       [[3, 0]],

       [[2, 0]],

       [[1, 0]]], dtype=int32),)


In [2]:
import argparse

import cv2


def get_opencv_result(video_to_process):
    # create VideoCapture object for further video processing
    captured_video = cv2.VideoCapture(video_to_process)
    # check video capture status
    if not captured_video.isOpened:
        print("Unable to open: " + video_to_process)
        exit(0)

    # instantiate background subtraction
    background_subtr_method = cv2.bgsegm.createBackgroundSubtractorGSOC()

    while True:
        # read video frames
        retval, frame = captured_video.read()

        # check whether the frames have been grabbed
        if not retval:
            break

        # resize video frames
        frame = cv2.resize(frame, (640, 360))

        # pass the frame to the background subtractor
        foreground_mask = background_subtr_method.apply(frame)
        # obtain the background without foreground mask
        background_img = background_subtr_method.getBackgroundImage()

        # show the current frame, foreground mask, subtracted result
        cv2.imshow("Initial Frames", frame)
        cv2.imshow("Foreground Masks", foreground_mask)
        cv2.imshow("Subtraction Result", background_img)

        keyboard = cv2.waitKey(10)
        if keyboard == 27:
            break


if __name__ == "__main__":
    # parser = argparse.ArgumentParser()
    # parser.add_argument(
    #     "--input_video",
    #     type=str,
    #     help="Define the full input video path",
    #     default="space_traffic.mp4",
    # )

    # # parse script arguments
    # args = parser.parse_args()

    input_video = "../space_traffic.mp4"

    # start BS-pipeline
    get_opencv_result(input_video)

: 